In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Deploy your containerized agent on Agent Runtime (prev. Agent Engine)

 <table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fagents%2Fagent_engine%2Ftutorial_deploy_your_containerised_agent.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/agents/agent_engine/tutorial_deploy_your_containerised_agent.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>


| Author |
| --- |
| [Hemanand Rajendran](https://github.com/heyanand) |

## Overview

[Agent Runtime](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime) (previously Agent Engine) offers a managed runtime environment designed for deploying, running, and scaling AI agents effectively. In a standard deployment, the platform automatically bundles your source code and package dependencies into a generic container image.

However, many enterprise workloads require complete ownership over the runtime environment. To support this requirement, Agent Runtime provides [**Bring Your Own Container (BYOC)**](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime/setup#byoc) capability, allowing developers to bypass the platform's built-in containerization step and deploy pre-built custom container images.

This notebook outlines the end-to-end process for containerizing an agent built with the Google Agent Development Kit (ADK), configuring the necessary Google Cloud platform prerequisites, and deploying it to Agent Runtime using the SDK.

## Get started

### Install the Python SDK
This step install the google-cloud-aiplatform package containing the modules for agent engine and the ADK framework.


In [ ]:
%pip install "google-cloud-aiplatform[agent_engines,adk]>=1.144"

### Authenticate your notebook environment

If you are running this notebook in **Google Colab**, run the cell below to authenticate your account.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project](https://docs.cloud.google.com/resource-manager/docs/creating-managing-projects) and a [development environment](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment).

In [ ]:
import os

import vertexai

# fmt: off
PROJECT_ID = ""  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "" # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
MODEL = "gemini-3.1-flash-lite" # @param {type: "string", placeholder: "[gemini-3.1-flash-lite]", isTemplate: true}
MODEL_REGION = "global" # @param {type: "string", placeholder: "[global]", isTemplate: true}
# fmt: on

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
if not LOCATION:
    LOCATION = os.environ.get("GOOGLE_CLOUD_REGION")


client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

## Source Files Setup

Create the directory for housing the agent code and the terraform templates

In [ ]:
! rm -rf ./weather-agent-code && mkdir ./weather-agent-code && mkdir ./weather-agent-code/weather_agent # location for the agent code

The following code defines the logic for a weather-retrieval agent and saves it to a file named `agent.py`.

Here's the breakdown:
<br>1. **Tool Definition (get_temperature)**: This function acts as a 'tool' for the agent. In this example, it uses the random library to simulate fetching a temperature for a specific location. It includes a docstring that the LLM uses to understand when and how to call the function.
<br>2. **Agent Initialization**: It creates an instance of the Agent class from the google.adk library.


In [ ]:
# Create a config.json file to store the configurations
import json

config_json = {
    "PROJECT_ID": f"{PROJECT_ID}",
    "MODEL": f"{MODEL}",
    "MODEL_REGION": f"{MODEL_REGION}",
    "LOCATION": f"{LOCATION}",
}

json.dump(config_json, open("./weather-agent-code/weather_agent/config.json", "w"))

In [ ]:
%%writefile ./weather-agent-code/weather_agent/agent.py
import json
import random
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini

from functools import cached_property
from google.genai import Client


llm_config = json.load(open("weather_agent/config.json"))

PROJECT_ID = llm_config["PROJECT_ID"]
MODEL = llm_config["MODEL"]
MODEL_REGION = llm_config["MODEL_REGION"]

# Overriding Gemini class to set location to global for models with Global endpoint.
class GlobalGemini(Gemini):
  @cached_property
  def api_client(self) -> Client:
    return Client(vertexai=True, location="global")

def get_temperature(place: str) -> str:
    '''Returns the current temperature of a given place.

    Args:
        place: The name of the city or location.

    Returns:
        str: A string describing the temperature.
    '''
    # In a real scenario, this would call a weather API.
    # For this sample, we'll return a simulated temperature.

    temp = random.randint(-10, 40)
    return f"The current temperature in {place} is {temp}°C."

llm_model = GlobalGemini(model=MODEL) if MODEL_REGION == "global" else Gemini(model=MODEL)

root_agent = Agent(
    model=llm_model,
    name='weather_agent',
    description='An agent that provides temperature information for locations.',
    instruction='You are a helpful assistant that can provide the current temperature for any given place using the get_temperature tool.',
    tools=[get_temperature],
)

In [ ]:
# Init file
%%writefile ./weather-agent-code/weather_agent/__init__.py

The following code included in the `main.py` sets up a FastAPI web server to act as a runtime for the AI agent. Here's a breakdown of what it does:

1. **FastAPI Setup**: It defines a
standard web application with a QueryRequest schema to handle incoming user inputs and specify which agent method to call<br>
2. **ADK Integration**: It wraps your root_agent using agent_engines.AdkApp, which provides the standard interface for interacting with agents built using the Agent Development Kit (ADK).<br>
3. **Endpoint** - /api/reasoning_engine: This is a standard POST endpoint that receives a query, executes the agent's logic, and returns a final JSON response.<br>
4. **Endpoint** - /api/stream_reasoning_engine: This handles streaming responses, allowing you to see the agent's thought process or partial answers in real-time as they are generated.


In [ ]:
%%writefile ./weather-agent-code/main.py
import inspect
import json
import logging
import os
import uvicorn
import vertexai
from weather_agent.agent import root_agent
from fastapi import FastAPI, encoders, responses
from pydantic import BaseModel
from vertexai import agent_engines

app = FastAPI()

config_json = json.load(open("weather_agent/config.json"))
PROJECT_ID = config_json["PROJECT_ID"]
LOCATION = config_json["LOCATION"]
MODEL_REGION = config_json["MODEL_REGION"]

class QueryRequest(BaseModel):
    input: dict | None = None
    class_method: str | None = None

vertexai.init(
    project=PROJECT_ID,
    location=MODEL_REGION,
)

adk_app = agent_engines.AdkApp(agent=root_agent)

def _encode_chunk_to_json(chunk):
  """Encodes a chunk to a JSON string with a newline."""
  try:
    json_chunk = encoders.jsonable_encoder(chunk)
    return json.dumps(json_chunk) + "\n"
  except Exception:
    logging.exception("Failed to encode chunk")
    return None

async def json_generator(output):
  async for chunk in output:
    encoded_chunk = _encode_chunk_to_json(chunk)
    if encoded_chunk is None:
      break
    yield encoded_chunk

async def _invoke_callable_or_raise(invocation_callable, invocation_payload):
  if inspect.iscoroutinefunction(invocation_callable):
    return await invocation_callable(**invocation_payload)
  else:
    return invocation_callable(**invocation_payload)

@app.post("/api/reasoning_engine")
async def query(request: QueryRequest) -> responses.JSONResponse:
    method = getattr(adk_app, request.class_method)
    output = await _invoke_callable_or_raise(method, request.input or {})

    try:
      json_serialized_content = encoders.jsonable_encoder({"output": output})
    except ValueError as encoding_error:
      logging.exception(
          "FastAPI could not JSON-encode the response from invocation method"
          " %s. Error: %s. Invocation method's original response: %r",
          request.class_method, encoding_error, output,
      )
      raise encoding_error
    return responses.JSONResponse(content=json_serialized_content)

@app.post("/api/stream_reasoning_engine")
async def stream_query(request: QueryRequest) -> responses.StreamingResponse:
    method = getattr(adk_app, request.class_method)
    output = await _invoke_callable_or_raise(method, request.input or {})
    return responses.StreamingResponse(
        content=json_generator(output),
        media_type="application/json",
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", 8080)))

Let's create the file containing the list of all dependencies needed to containerize the agent.

In [ ]:
%%writefile ./weather-agent-code/requirements.txt
fastapi
uvicorn
vertexai
google-cloud-aiplatform[agent_engines,adk]>=1.144
pydantic

The following cell creates a `Dockerfile`, which contains the instructions to build a container image for the agent.

In [ ]:
%%writefile ./weather-agent-code/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY weather_agent/ /app/weather_agent/
COPY main.py .
COPY requirements.txt .
RUN pip install -r requirements.txt

CMD ["sh", "-c", "uvicorn main:app --host 0.0.0.0 --port $PORT"]

## Container Build

Let's create a repository in Google Artifact Registry to save the container images containing the agent code.

In [ ]:
!gcloud auth login

In [ ]:
!gcloud services enable cloudbuild.googleapis.com compute.googleapis.com artifactregistry.googleapis.com --project=$PROJECT_ID

In [ ]:
REPOSITORY_NAME = "agents-repo"

In [ ]:
!gcloud artifacts repositories create $REPOSITORY_NAME \
    --project=$PROJECT_ID \
    --repository-format=docker \
    --location=$LOCATION \
    --description="Docker repository"

Let's validate if the repository has been successfully created.

In [ ]:
!gcloud artifacts repositories list --project=$PROJECT_ID --location=$LOCATION

In [ ]:
PROJECT_NUMBER = !gcloud projects describe $PROJECT_ID --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]
PROJECT_NUMBER

Grant the Artifact Registry Writer role (`roles/artifactregistry.writer`) and Storage Object Viewer (`roles/storage.objectViewer`) to the project's **Default Compute Service Account**. This permission is necessary so that when we build the agent container (using Cloud Build or local tools), the service account has the authority to **upload** (push) the resulting Docker image to the Artifact Registry repository.

In [ ]:
!gcloud projects add-iam-policy-binding $PROJECT_ID \
    --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com" \
    --role="roles/artifactregistry.writer" \
    --condition=None

In [ ]:
!gcloud projects add-iam-policy-binding $PROJECT_NUMBER \
    --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com" \
    --role="roles/storage.objectViewer"  \
    --condition=None

Let's grant the Artifact Registry permissions to the Reasoning Engine Service Agent and the AI Platform Service Agent

In [ ]:
!gcloud projects add-iam-policy-binding $PROJECT_NUMBER \
    --member="serviceAccount:service-$PROJECT_NUMBER@gcp-sa-aiplatform-re.iam.gserviceaccount.com" \
    --role="roles/artifactregistry.reader"  --condition=None
!gcloud projects add-iam-policy-binding $PROJECT_NUMBER \
    --member="serviceAccount:service-$PROJECT_NUMBER@gcp-sa-aiplatform.iam.gserviceaccount.com" \
    --role="roles/artifactregistry.reader"  --condition=None

Let's use the Dockerfile to build the container image using Cloud Build. At the end of the build process, the container image is pushed to the Artifact Registry repository created earlier.

In [ ]:
!gcloud config set project {PROJECT_ID}
!gcloud builds submit \
    --project={PROJECT_ID} \
    --region={LOCATION} \
    --tag {LOCATION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY_NAME}/weather-agent-image:latest \
    ./weather-agent-code

### Tenant Service Account - Identification & Permissions

A tenant service account (or tenant service agent) is a managed service account created and maintained by Google to perform actions on your behalf within a tenant project. It allows managed Google services to interact with your resources securely without requiring manual user credentials.<br>

In this section, let's identify the tenant service account and grant the necessary permissions needed to access the container image from the Artifact Registry for the containerized agent deployment.


In [ ]:
!mkdir -p tenant_sa_setup

In [ ]:
%%writefile tenant_sa_setup/requirements.txt
google-cloud-aiplatform[agent_engines]

In [ ]:
%%writefile tenant_sa_setup/my_agent.py
class MetadataAgent:

  def query(self):
    import requests
    url = "http://metadata.google.internal/computeMetadata/v1/project/numeric-project-id"
    try:
        response = requests.get(url, headers={"Metadata-Flavor": "Google"})
        response.raise_for_status()
        return f"service-{response.text}@serverless-robot-prod.iam.gserviceaccount.com"
    except Exception:
        return None

root_agent = MetadataAgent()

In [ ]:
tenant_sa_agent = client.agent_engines.create(
    config={
        "source_packages": ["tenant_sa_setup"],
        "entrypoint_module": "tenant_sa_setup.my_agent",
        "entrypoint_object": "root_agent",
        "requirements_file": "tenant_sa_setup/requirements.txt",
        "class_methods": [{"api_mode": "", "name": "query"}],
    },
)

tenant_sa_agent

Let's copy the value of the tenant service account indicated by the cell containing `tenant_sa_agent.query()` into the variable `TENANT_SERVICE_ACCOUNT` present in the cell below.

In [ ]:
# Your tenant service account in the format "service-{number}@serverless-robot-prod.iam.gserviceaccount.com"
TENANT_SERVICE_ACCOUNT = tenant_sa_agent.query()

In [ ]:
# Delete the agent.
tenant_sa_agent.delete(force=True)

Grant the Artifact Registry reader role to the Tenant Service Account.

In [ ]:
!gcloud projects add-iam-policy-binding $PROJECT_NUMBER \
    --member="serviceAccount:$TENANT_SERVICE_ACCOUNT" \
    --role="roles/artifactregistry.reader"  --condition=None

❗ Additionally, if your Organization enforces the `iam.allowedPolicyMemberDomains` organization policy constraint, you need to allow the Google's customer domain before adding the permissions to the Tenant Service account. For detailed instructions, check [Tenant Service Account Setup Documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/runtime/setup#tenant_service_account)

## Deploy the agent with SDK

This code deploys your custom containerized agent to Agent Runtime. Here's the breakdown:


1. **client.agent_engines.create**: This method registers and deploys the agent on Google Cloud.
2. **display_name & description**: Sets the metadata for the agent as it will appear in the Google Cloud Console.
3. **container_spec**: This is the 'Bring Your Own Container' (BYOC) part. It points to the Docker image you previously built and pushed to Artifact Registry using the image_uri.
4. **class_methods**: This list defines the API interface for your agent. It maps standard agent actions (like creating sessions or streaming queries) to the methods available in your code, supporting both synchronous and asynchronous modes.
5. **agent_framework**: Setting this to 'google-adk' tells the platform that your agent is built using the Google Agent Development Kit, which enables features like the playground in the Google Cloud Platform console.


After execution, the `remote_agent` holds a reference to the deployed agent, allowing us to interact with it via the SDK.

In [ ]:
remote_agent = client.agent_engines.create(
    config={
        "display_name": "byoc_weather_agent",
        "description": "byoc testing with example agent",
        "container_spec": {
            "image_uri": f"{LOCATION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY_NAME}/weather-agent-image:latest"
        },
        "class_methods": [
            # For convenience to interact with the agent through the Python SDK
            # https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/use-an-adk-agent
            {"api_mode": "", "name": "get_session"},
            {"api_mode": "", "name": "list_sessions"},
            {"api_mode": "", "name": "create_session"},
            {"api_mode": "", "name": "delete_session"},
            {"api_mode": "async", "name": "async_get_session"},
            {"api_mode": "async", "name": "async_list_sessions"},
            {"api_mode": "async", "name": "async_create_session"},
            {"api_mode": "async", "name": "async_delete_session"},
            {"api_mode": "async", "name": "async_add_session_to_memory"},
            {"api_mode": "async", "name": "async_search_memory"},
            {"api_mode": "stream", "name": "stream_query"},
            {"api_mode": "async_stream", "name": "async_stream_query"},
            {"api_mode": "async_stream", "name": "streaming_agent_run_with_events"},
        ],
        "agent_framework": "google-adk",  # For usage through the playground on Google Cloud console
    },
)
remote_agent

## Query

Let's test if the agent is responding to our queries. We will be querying the stream_query API of the remote agent.

In [ ]:
import json

import requests
from google import auth as google_auth
from google.auth.transport import requests as google_requests

def get_identity_token():
    credentials, _ = google_auth.default()
    auth_request = google_requests.Request()
    credentials.refresh(auth_request)
    return credentials.token


response = requests.post(
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/{remote_agent.api_resource.name}:streamQuery",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {get_identity_token()}",
    },
    data=json.dumps(
        {
            "class_method": "async_stream_query",
            "input": {
                "user_id": "test_user_123",
                "message": "What is the temperature in New Delhi?",
            },
        }
    ),
    stream=True,
)
for chunk in response.iter_content(chunk_size=8192):
    print(chunk)

## Cleaning up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the individual resources you created in this tutorial by running the cell below.

In [ ]:
# 1. Delete the Reasoning Engines (Agents)
# Note: We retrieve the list first to ensure we delete the ones created in this session
try:
    page_size = 100
    reasoning_engines = client.agent_engines.list()
    for engine in reasoning_engines:
        if (
            "weather_agent" in engine.api_resource.display_name
            or "MetadataAgent" in engine.api_resource.display_name
        ):
            print(f"Deleting Reasoning Engine: {engine.api_resource.name}")
            engine.delete(force=True)
except Exception as e:
    print(f"Error deleting reasoning engines: {e}")

# 2. Delete the Artifact Registry Repository
!gcloud artifacts repositories delete $REPOSITORY_NAME --location=$LOCATION --project=$PROJECT_ID --quiet

# 3. Clean up local directories
!rm -rf ./weather-agent-code ./tenant_sa_setup

print("Cleanup of specific resources completed.")